In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import pickle
import os
import warnings
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import shap
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold
import catboost as cb

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
# importing files preprocessed
X_train  = pd.read_csv("preprocessed_data/X_train_pp.csv")
X_test  = pd.read_csv("preprocessed_data/X_test_pp.csv")
X_val  = pd.read_csv("preprocessed_data/X_val_pp.csv")
y_train_log  = pd.read_csv("preprocessed_data/y_train_log.csv")
y_test_log  = pd.read_csv("preprocessed_data/y_test_log.csv")
y_val_log  = pd.read_csv("preprocessed_data/y_val_log.csv")
y_train_raw  = pd.read_csv("preprocessed_data/y_train_raw.csv")
y_test_raw  = pd.read_csv("preprocessed_data/y_test_raw.csv")
y_val_raw  = pd.read_csv("preprocessed_data/y_val_raw.csv")
non_agri_train = pd.read_csv("preprocessed_data/non_agri_train.csv")
non_agri_test = pd.read_csv("preprocessed_data/non_agri_test.csv")
non_agri_val = pd.read_csv("preprocessed_data/non_agri_val.csv")

In [ ]:
farmer_ids = pd.read_csv("final_data/farmer_id_final.csv")
X_final= pd.read_csv("final_data/X_final_pp.csv")

###FEATURE SELECTION- REMOVING LOW VARIANCE, HIGHLY COLLINEAR FEATURES AND RANDOM FOREST BEST FEATURE SELECTOR AND A SHAP SELECTOR

In [ ]:
class LowVarianceRemover(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold
        self.cols_to_keep_ = []

    def fit(self, X, y=None):
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        
        variances = X[numeric_cols].var()
        high_var = variances[variances > self.threshold].index.tolist()
        
        # keep all non-numeric columns as-is
        non_numeric = [c for c in X.columns if c not in numeric_cols]
        self.cols_to_keep_ = non_numeric + high_var
        
        dropped = [c for c in numeric_cols if c not in high_var]
        print(f"Low variance filter — dropped {len(dropped)}, kept {len(self.cols_to_keep_)} features")
        if dropped:
            print(f"  Dropped: {dropped}")
        
        return self

    def transform(self, X):
        return X[[c for c in self.cols_to_keep_ if c in X.columns]]

In [ ]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.90):
        self.threshold = threshold
        self.cols_to_keep_ = []

    def fit(self, X, y=None):
        corr  = X.select_dtypes(include=[np.number]).corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

        to_drop = []
        for col in upper.columns:
            high = upper[col][upper[col] > self.threshold]
            for corr_col in high.index:
                if corr_col not in to_drop:
                    to_drop.append(corr_col)

        self.cols_to_keep_ = [c for c in X.columns if c not in to_drop]
        print(f"Correlation filter — kept {len(self.cols_to_keep_)} / {X.shape[1]} features")
        return self

    def transform(self, X):
        return X[[c for c in self.cols_to_keep_ if c in X.columns]]

In [ ]:
class RandomForestSelector(BaseEstimator, TransformerMixin):
    def __init__(self, n_features=30, n_estimators=200, random_state=42):
        self.n_features   = n_features
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.selected_cols_  = []
        self.importances_    = None

    def fit(self, X, y):
        rf = RandomForestRegressor(
            n_estimators=self.n_estimators,
            max_depth=10,
            min_samples_leaf=20,
            random_state=self.random_state,
            n_jobs=-1
        )
        rf.fit(X, y)

        self.importances_ = pd.Series(
            rf.feature_importances_, index=X.columns
        ).sort_values(ascending=False)

        self.selected_cols_ = self.importances_.head(self.n_features).index.tolist()

        print(f"\nTop {self.n_features} features selected by Random Forest:")
        print(self.importances_.head(self.n_features).to_string())
        return self

    def transform(self, X):
        return X[[c for c in self.selected_cols_ if c in X.columns]]

In [ ]:
class SHAPSelector(BaseEstimator, TransformerMixin):
    def __init__(self, n_features=30, n_estimators=300, random_state=42, val_X=None):
        self.n_features   = n_features
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.val_X        = val_X
        self.selected_cols_ = []
        self.importances_   = None
        self._col_mapping   = {}   # original -> cleaned

    def _clean_names(self, X):
        import re
        new_cols = {}
        for col in X.columns:
            new = re.sub(r'[^\w]', '_', col.strip())
            new = re.sub(r'_+', '_', new).strip('_')
            new_cols[col] = new
        self._col_mapping = new_cols
        return X.rename(columns=new_cols)

    def fit(self, X, y):
        X_clean = self._clean_names(X)

        eval_X = self._clean_names(self.val_X) if self.val_X is not None else X_clean

        model = lgb.LGBMRegressor(
            n_estimators=self.n_estimators,
            random_state=self.random_state,
            verbose=-1,
            n_jobs=-1
        )
        model.fit(X_clean, y)

        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(eval_X)

        # map importance back to original column names
        reverse_mapping = {v: k for k, v in self._col_mapping.items()}
        importances_clean = pd.Series(
            np.abs(shap_vals).mean(axis=0),
            index=eval_X.columns
        ).sort_values(ascending=False)

        self.importances_ = importances_clean.rename(index=reverse_mapping)
        self.selected_cols_ = self.importances_.head(self.n_features).index.tolist()

        print(f"\n[SHAPSelector] Top {self.n_features} features:")
        print(self.importances_.head(self.n_features).to_string())
        return self

    def transform(self, X):
        return X[[c for c in self.selected_cols_ if c in X.columns]]

In [ ]:
xgb_pipeline = Pipeline([
    ('lvr', LowVarianceRemover(threshold=0.01)),  
    ('corr',   CorrelationFilter(threshold=0.90)),
    ('shap', SHAPSelector(n_features=40))
])

X_train_xgb = xgb_pipeline.fit_transform(X_train, y_train_log)
X_val_xgb   = xgb_pipeline.transform(X_val)
X_test_xgb  = xgb_pipeline.transform(X_test)
X_final_xgb = xgb_pipeline.transform(X_final)

In [ ]:
cb_pipeline = Pipeline([
    ('lvr', LowVarianceRemover(threshold=0.01)),  
    ('corr',   CorrelationFilter(threshold=0.90)),
])

X_train_cb = cb_pipeline.fit_transform(X_train, y_train_log)
X_val_cb   = cb_pipeline.transform(X_val)
X_test_cb  = cb_pipeline.transform(X_test)
X_final_cb = cb_pipeline.transform(X_final)

WRITING MAPE FUNCTIONS

In [ ]:
def mape(y_true, y_pred):
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

def total_income_mape(y_agri_true, y_agri_pred, non_agri):
    total_true = np.array(y_agri_true).ravel() + np.array(non_agri).ravel()
    total_pred = np.array(y_agri_pred).ravel() + np.array(non_agri).ravel()
    return mape(total_true, total_pred)

CATBOOST

In [ ]:
X_full_cb     = pd.concat([X_train_cb, X_val_cb], axis=0).reset_index(drop=True)
y_full_log    = pd.concat([y_train_log, y_val_log], axis=0).reset_index(drop=True)
y_full_raw    = pd.concat([y_train_raw, y_val_raw], axis=0).reset_index(drop=True)

def objective_cat(trial):
    grow_policy = trial.suggest_categorical(
        'grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide'])
    params = {
        'eval_metric'         : 'MAE',
        'verbose'             : False,
        'thread_count'        : -1,
        'boosting_type'       : trial.suggest_categorical('boosting_type', ['Ordered', 'Plain'])
                                if grow_policy == 'SymmetricTree' else 'Plain',
        'learning_rate'       : trial.suggest_float('learning_rate', 0.003, 0.2,  log=True),
        'iterations'          : trial.suggest_int('iterations',   500, 3000),
        'depth'               : trial.suggest_int('depth',          3,   10),
        'subsample'           : trial.suggest_float('subsample',    0.4,  1.0),
        'rsm'                 : trial.suggest_float('rsm',          0.3,  1.0),
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    maes = []
    for train_idx, val_idx in kf.split(X_full_cb):
        model = cb.CatBoostRegressor(**params)
        model.fit(
            X_full_cb.iloc[train_idx], y_full_log.iloc[train_idx],
            eval_set=(X_full_cb.iloc[val_idx], y_full_log.iloc[val_idx]),
            early_stopping_rounds=50, verbose=False
        )
        pred_raw = np.expm1(model.predict(X_full_cb.iloc[val_idx]))
        maes.append(mean_absolute_error(y_full_raw.iloc[val_idx], pred_raw))
    return np.mean(maes)

study_cat = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
)
study_cat.optimize(objective_cat, n_trials=15, show_progress_bar=True)
print(f"CatBoost best CV MAE: {study_cat.best_value:.2f}")

cat_best = {
    **study_cat.best_params,
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'random_seed': 42, 'verbose': False, 'thread_count': -1,
}
if cat_best.get('grow_policy', 'SymmetricTree') != 'SymmetricTree':
    cat_best['boosting_type'] = 'Plain'

ref_cat = cb.CatBoostRegressor(**cat_best)
ref_cat.fit(X_train_cb, y_train_log,
            eval_set=(X_val_cb, y_val_log),
            early_stopping_rounds=100, verbose=False)
best_iter_cat = ref_cat.best_iteration_
print(f"CatBoost best iteration: {best_iter_cat}")

final_cat = cb.CatBoostRegressor(**{**cat_best, 'iterations': best_iter_cat})
final_cat.fit(X_full_cb, y_full_log, verbose=False)

cat_val_pred  = np.expm1(final_cat.predict(X_val_cb))
cat_test_pred = np.expm1(final_cat.predict(X_test_cb))


XGBOOST

In [ ]:

dtrain = xgb.DMatrix(X_train_xgb, label=y_train_log)
dval   = xgb.DMatrix(X_val_xgb,   label=y_val_log)
dtest  = xgb.DMatrix(X_test_xgb,  label=y_test_log)

def objective(trial):
    params = {
        'objective'         : 'reg:squarederror',
        'tree_method'       : 'hist',
        'verbosity'         : 0,
        'seed'              : 42,
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 3, 6),      # was 3-10
        'min_child_weight' : trial.suggest_float('min_child_weight', 10, 100, log=True),  # was 1-50
        'subsample'        : trial.suggest_float('subsample', 0.5, 0.8),  # was 0.5-1.0
        'gamma'             : trial.suggest_float('gamma', 0.0, 5.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel' : trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'colsample_bynode'  : trial.suggest_float('colsample_bynode', 0.4, 1.0),
        'lambda'            : trial.suggest_float('lambda', 1e-3, 20.0, log=True),
        'alpha'             : trial.suggest_float('alpha', 1e-3, 20.0, log=True),
        'max_bin'           : trial.suggest_int('max_bin', 128, 512),
    }

    model = xgb.train(
        params, dtrain,
        num_boost_round=5000,
        evals=[(dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    val_pred_raw = np.expm1(model.predict(dval))
    return mean_absolute_error(y_val_raw, val_pred_raw)


study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=10)
)

print("\nTuning — 100 trials...")
study.optimize(objective, n_trials=80, show_progress_bar=True)

print(f"\nBest val MAE : {study.best_value:.2f}")
print("Best params  :")
for k, v in study.best_params.items():
    print(f"  {k:25s}: {v}")

best_params = {
    **study.best_params,
    'objective'  : 'reg:squarederror',
    'tree_method': 'hist',
    'verbosity'  : 0,
    'seed'       : 42,
}

ref_model = xgb.train(
    best_params, dtrain,
    num_boost_round=5000,
    evals=[(dval, 'val')],
    early_stopping_rounds=100,
    verbose_eval=False
)
best_rounds = ref_model.best_iteration
print(f"\nBest iteration: {best_rounds}")

X_tv = pd.concat([X_train_xgb, X_val_xgb], axis=0).reset_index(drop=True)
y_tv = pd.concat([y_train_log, y_val_log], axis=0).reset_index(drop=True)
non_agri_tv = pd.concat([non_agri_train, non_agri_val], axis=0).reset_index(drop=True)

dtv = xgb.DMatrix(X_tv, label=y_tv)

final_model = xgb.train(
    best_params, dtv,
    num_boost_round=best_rounds,
    verbose_eval=False
)


val_pred_raw  = np.expm1(final_model.predict(dval))
test_pred_raw = np.expm1(final_model.predict(dtest))

print("\n" + "=" * 55)
print("FINAL RESULTS")
print("=" * 55)
print(f"  Val  MAE          : {mean_absolute_error(y_val_raw,  val_pred_raw):.2f}")
print(f"  Test MAE          : {mean_absolute_error(y_test_raw, test_pred_raw):.2f}")
print(f"  Val  MAPE (agri)  : {mape(y_val_raw,  val_pred_raw):.4f}%")
print(f"  Test MAPE (agri)  : {mape(y_test_raw, test_pred_raw):.4f}%")
print(f"  Val  MAPE (total) : {total_income_mape(y_val_raw,  val_pred_raw, non_agri_val):.4f}%")
print(f"  Test MAPE (total) : {total_income_mape(y_test_raw, test_pred_raw, non_agri_test):.4f}%")
print("=" * 55)


ENSEMBLE LEARNING 

In [ ]:
mae_cat = mean_absolute_error(y_val_raw, cat_val_pred)
mae_xgb = mean_absolute_error(y_val_raw, val_pred_raw)
w_cat   = (1/mae_cat) / (1/mae_cat + 1/mae_xgb)
w_xgb   = (1/mae_xgb) / (1/mae_cat + 1/mae_xgb)

ensemble_val_pred  = w_cat * cat_val_pred  + w_xgb * val_pred_raw
ensemble_test_pred = w_cat * cat_test_pred + w_xgb * test_pred_raw

print(f"\nWeights — CatBoost: {w_cat:.3f}, XGBoost: {w_xgb:.3f}")

print("\n" + "=" * 55)
print("FINAL RESULTS")
print("=" * 55)
for name, val_p, test_p in [
    ('CatBoost',  cat_val_pred,      cat_test_pred),
    ('XGBoost',   val_pred_raw,      test_pred_raw),
    ('Ensemble',  ensemble_val_pred, ensemble_test_pred)
]:
    print(f"\n  {name}")
    print(f"    Val  MAE          : {mean_absolute_error(y_val_raw,  val_p):.2f}")
    print(f"    Test MAE          : {mean_absolute_error(y_test_raw, test_p):.2f}")
    print(f"    Val  MAPE (agri)  : {mape(y_val_raw,  val_p):.4f}%")
    print(f"    Test MAPE (agri)  : {mape(y_test_raw, test_p):.4f}%")
    print(f"    Val  MAPE (total) : {total_income_mape(y_val_raw,  val_p, non_agri_val):.4f}%")
    print(f"    Test MAPE (total) : {total_income_mape(y_test_raw, test_p, non_agri_test):.4f}%")
print("=" * 55)

In [ ]:
cat_final_pred = np.expm1(final_cat.predict(X_final_cb))
xgb_final_pred = np.expm1(final_model.predict(xgb.DMatrix(X_final_xgb)))

In [ ]:

def ensure_1d(arr):
    """Convert any array to 1D"""
    arr = np.array(arr)
    if len(arr.shape) > 1:
        arr = arr.ravel()
    return arr


cat_final_pred = ensure_1d(cat_final_pred)
xgb_final_pred = ensure_1d(xgb_final_pred)
farmer_ids = ensure_1d(farmer_ids)


ensemble_final_pred = w_cat * cat_final_pred + w_xgb * xgb_final_pred


assert len(ensemble_final_pred.shape) == 1, f"Shape is {ensemble_final_pred.shape}, should be 1D"


submission = pd.DataFrame({
    'FarmerID': farmer_ids,
    'Total_Income': ensemble_final_pred
})

submission.to_csv("submission.csv", index=False)
print(submission.head())